In [ ]:
from google.colab import drive
import sys

drive.mount('/content/drive')

sys.path.append('/content/drive/MyDrive/Models/ArQModel')


In [ ]:
import os
import json
import pickle
import numpy as np
import torch
from tqdm import tqdm
from collections import Counter
from random import choice, sample


def create_input_files(dataset, karpathy_json_path, captions_per_image, min_word_freq, output_folder, max_len=100):
    assert dataset == 'coco'
    os.makedirs(output_folder, exist_ok=True)

    with open(karpathy_json_path, 'r', encoding='utf-8') as j:
        data = json.load(j)

    train_pkl = os.path.join(output_folder, 'train36_imgid2idx.pkl')
    val_pkl = os.path.join(output_folder, 'val36_imgid2idx.pkl')
    if not os.path.exists(train_pkl) or not os.path.exists(val_pkl):
        raise FileNotFoundError("Missing train36_imgid2idx.pkl or val36_imgid2idx.pkl in output folder.")

    with open(train_pkl, 'rb') as f:
        train_data = pickle.load(f)
    with open(val_pkl, 'rb') as f:
        val_data = pickle.load(f)

    train_caps, val_caps, test_caps = [], [], []
    train_dets, val_dets, test_dets = [], [], []
    word_freq = Counter()

    for img in tqdm(data['images']):
        caps = []
        for c in img['sentences']:
            word_freq.update(c['tokens'])
            if len(c['tokens']) <= max_len:
                caps.append(c['tokens'])
        if not caps:
            continue

        image_id = str(int(img['filename'].split('_')[2].split('.')[0]))
        split = img['split'].lower()

        if split in {'train', 'restval'}:
            idx = train_data.get(image_id) or val_data.get(image_id)
            if idx is not None:
                train_dets.append(idx)
                train_caps.append(caps)
        elif split == 'val':
            idx = val_data.get(image_id) or train_data.get(image_id)
            if idx is not None:
                val_dets.append(idx)
                val_caps.append(caps)
        elif split == 'test':
            idx = val_data.get(image_id) or train_data.get(image_id)
            if idx is not None:
                test_dets.append(idx)
                test_caps.append(caps)

    words = [w for w in word_freq.keys() if word_freq[w] > min_word_freq]
    word_map = {k: v + 1 for v, k in enumerate(words)}
    word_map['<unk>'] = len(word_map) + 1
    word_map['<start>'] = len(word_map) + 1
    word_map['<end>'] = len(word_map) + 1
    word_map['<pad>'] = 0

    base_filename = f"{dataset}_{captions_per_image}_cap_per_img_{min_word_freq}_min_word_freq"
    with open(os.path.join(output_folder, f"WORDMAP_{base_filename}.json"), 'w', encoding='utf-8') as j:
        #json.dump(word_map, j)
        json.dump(word_map, j ,ensure_ascii=False, indent=2)


    for split_name, dets, caps_list in [('TRAIN', train_dets, train_caps),
                                        ('VAL', val_dets, val_caps),
                                        ('TEST', test_dets, test_caps)]:
        encoded_caps = []
        caplens = []
        for caps in caps_list:
            if len(caps) < captions_per_image:
                sampled = caps + [choice(caps) for _ in range(captions_per_image - len(caps))]
            else:
                sampled = sample(caps, k=captions_per_image)

            for c in sampled:
                enc_c = [word_map['<start>']] + [word_map.get(w, word_map['<unk>']) for w in c][:max_len - 2] + [word_map['<end>']]
                c_len = len(enc_c)
                if c_len < max_len:
                    enc_c += [word_map['<pad>']] * (max_len - c_len)
                encoded_caps.append(enc_c)
                caplens.append(c_len)

        with open(os.path.join(output_folder, f"{split_name}_CAPTIONS_{base_filename}.json"), 'w') as j:
            json.dump(encoded_caps, j)
        with open(os.path.join(output_folder, f"{split_name}_CAPLENS_{base_filename}.json"), 'w') as j:
            json.dump(caplens, j)

    with open(os.path.join(output_folder, f"TRAIN_GENOME_DETS_{base_filename}.json"), 'w') as j:
        json.dump(train_dets, j)
    with open(os.path.join(output_folder, f"VAL_GENOME_DETS_{base_filename}.json"), 'w') as j:
        json.dump(val_dets, j)
    with open(os.path.join(output_folder, f"TEST_GENOME_DETS_{base_filename}.json"), 'w') as j:
        json.dump(test_dets, j)


def save_checkpoint(data_name, epoch, epochs_since_improvement, decoder, decoder_optimizer, bleu4, is_best):
    """
    Save a safe and flexible checkpoint compatible with both training and evaluation.
    """

    checkpoint_dir = '/content/drive/MyDrive/Models/ArQModel/Checkpoints'
    os.makedirs(checkpoint_dir, exist_ok=True)

    # Save both the full model and the state_dict for compatibility
    state = {
        'epoch': epoch,
        'epochs_since_improvement': epochs_since_improvement,
        'bleu-4': float(bleu4),
        'decoder': decoder,  # full model (useful for evaluation)
        'decoder_state_dict': decoder.state_dict(),  # lightweight version (for training resume)
        'decoder_optimizer_state_dict': decoder_optimizer.state_dict(),
    }

    latest_path = os.path.join(checkpoint_dir, f'{data_name}_checkpoint.pth.tar')
    torch.save(state, latest_path)

    epoch_path = os.path.join(checkpoint_dir, f'{data_name}_epoch_{epoch:03d}.pth.tar')
    torch.save(state, epoch_path)

    if is_best:
        best_path = os.path.join(checkpoint_dir, f'{data_name}_BEST.pth.tar')
        torch.save(state, best_path)
        print(f"Saved BEST checkpoint at epoch {epoch} (BLEU-4: {bleu4:.4f})")

    print(f"Checkpoint saved at: {latest_path}")



class AverageMeter(object):
    """Keeps track of average, sum, and count of a metric."""

    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def adjust_learning_rate(optimizer, shrink_factor):
    """Decays learning rate by a given factor."""
    print("\nDecaying learning rate...")
    for param_group in optimizer.param_groups:
        param_group['lr'] *= shrink_factor
    print(f"New learning rate: {param_group['lr']:.6f}\n")


def accuracy(scores, targets, k):
    """Computes top-k accuracy."""
    batch_size = targets.size(0)
    _, ind = scores.topk(k, 1, True, True)
    correct = ind.eq(targets.view(-1, 1).expand_as(ind))
    correct_total = correct.view(-1).float().sum()
    return correct_total.item() * (100.0 / batch_size)
